In [69]:
# Imports

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.tuner import Tuner
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Getting Data

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadmusharraf444/used-car-price-prediction-dataset")
data_path = os.path.join(path, "quikr_car.csv")

print("Path to dataset files:", data_path)

Path to dataset files: /home/franio/.cache/kagglehub/datasets/muhammadmusharraf444/used-car-price-prediction-dataset/versions/1/quikr_car.csv


In [36]:
data_df = pd.read_csv(data_path)

data_df.dropna(inplace=True)
data_df = data_df[data_df.Price != 'Ask For Price' ]
data_df.Price = data_df.Price.str.replace(',', '').astype(float)

data_df.kms_driven = data_df.kms_driven.str.strip('kms').str.replace(',', '', regex=False).str.strip().astype(float)
data_df.year = data_df.year.astype(int)
data_df = pd.get_dummies(data_df, columns=['company', 'fuel_type'], dtype=int)

data_df.drop(['name'], axis=1, inplace=True)

data_df

,year,Price,kms_driven,company_Audi,company_BMW,company_Chevrolet,company_Datsun,company_Fiat,company_Force,company_Ford,...,company_Nissan,company_Renault,company_Skoda,company_Tata,company_Toyota,company_Volkswagen,company_Volvo,fuel_type_Diesel,fuel_type_LPG,fuel_type_Petrol
0,2007,80000.0,45000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2006,425000.0,40.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,2014,325000.0,28000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,2014,575000.0,36000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
6,2012,175000.0,41000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2011,270000.0,50000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
885,2009,110000.0,30000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
886,2009,300000.0,132000.0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
888,2018,260000.0,27000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0


# Classes

In [97]:
class MyDataset(Dataset):
 def __init__(self, data, outputs):
  """Inicjalizacja - przygotowanie struktury danych"""
  self.data = data
  self.outputs = outputs

 def __len__(self):
  """Zwraca liczbę próbek w datasecie"""
  return len(self.data)

 def __getitem__(self, idx):
  """Zwraca próbkę o indeksie idx"""
  x = self.data[idx]
  y = self.outputs[idx]
  return x, y

class MyDataModule(pl.LightningDataModule):
    def __init__(self,X:np.ndarray, y:float, batch_size=32):
        super().__init__()
        self.X = X
        self.y = y
        self.X_scaler = StandardScaler()
        self.y_scaler = StandardScaler()
        self.batch_size = batch_size


    def setup(self, stage=None):
        X_train_val, X_test, y_train_val, y_test = train_test_split(self.X, self.y, test_size=0.1)


        # Data scaling
        X_train_val = self.X_scaler.fit_transform(X_train_val)
        y_train_val = self.y_scaler.fit_transform(y_train_val.reshape(-1, 1)).ravel()
        X_test = self.X_scaler.transform(X_test)
        y_test = self.y_scaler.transform(y_test.reshape(-1, 1)).ravel()

        X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)

        self.train_dataset = MyDataset(
                torch.from_numpy(X_train).float(),
                torch.from_numpy(y_train).float()
            )
        self.val_dataset = MyDataset(
                torch.from_numpy(X_val).float(),
                torch.from_numpy(y_val).float()
            )
        self.test_dataset = MyDataset(
                torch.from_numpy(X_test).float(),
                torch.from_numpy(y_test).float()
            )

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [99]:
class LitModel(pl.LightningModule):
 def __init__(self, input_size,n_hidden, hidden_size, output_dim,lr=0.001):
    super().__init__()
    self.save_hyperparameters() # Call this first to save __init__ arguments

    # to self.hparams
    self.input_layer = nn.Linear(input_size, hidden_size)
    self.hidden_layers = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_hidden)])
    self.output_layer = nn.Linear(hidden_size, output_dim)
    self.criterion = nn.MSELoss()

 def forward(self, x):
    x = torch.relu(self.input_layer(x))
    for layer in self.hidden_layers:
        x = torch.relu(layer(x))

    x = self.output_layer(x)
    return x

 def training_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  self.log('train_loss', loss)
  return loss

 def validation_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('val_loss', loss, prog_bar=True)
  self.log('val_r2', r2, prog_bar=True)

 def test_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('test_loss', loss)
  self.log('test_r2', r2)

 def configure_optimizers(self):
  # Use the learning rate from self.hparams
  return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# Optimization

In [100]:
def train_model(config, X, y):
    model = LitModel(input_size=X.shape[1],
        n_hidden=config['n_hidden'],
        hidden_size=config['hidden_size'],
        output_dim=1,
        lr=config['lr']
    )

    datamodule = MyDataModule(X=X, y=y, batch_size=8)

    trainer = Trainer(
        max_epochs=20,
        log_every_n_steps=5,
        enable_checkpointing=False,
        callbacks=[TuneReportCallback({'loss': 'val_loss'}, on='validation_end')],
    )

    trainer.fit(model, datamodule)

# Ray Tune search
trainable_with_data = tune.with_parameters(train_model, X=X, y=y)

analysis = tune.run(
    trainable_with_data,
    config={
        'n_hidden': tune.choice([3,6,8,10]),
        'hidden_size': tune.choice([32, 64, 128, 256, 512]),
        'lr': tune.loguniform(1e-4, 5e-1),
    },
    num_samples=20
)

2026-06-15 13:54:00,959	INFO tune.py:615 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(train_model pid=30764) /home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(train_model pid=30753) GPU available: False, used: False
(train_model pid=30753) TPU available: False, using: 0 TPU cores
(train_model pid=30753) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
(train_model pid=30753) 
(train_model pid=30753)   | Name          | Type       | Params | Mode  | FLOPs
(train_model pid=30753) -------------------------------------------------------------
(train_model pid=30753) 0 | input_layer   | Linear     | 7.9 K  | train | 0    
(train_model pid=30753) 1 | hidden_layers | Modul

Epoch 0:   0%|          | 0/74 [00:00<?, ?it/s]                            
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]


(train_model pid=30764) 
(train_model pid=30753) /home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
(train_model pid=30753) /home/franio/Desktop/Deep_Learning_Labs/venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
(train_model pid=30758) 
(train_model pid=30761) 


Epoch 0:   9%|▉         | 7/74 [00:00<00:02, 31.00it/s, v_num=0]


...

(train_model pid=31446) `Trainer.fit` stopped: `max_epochs=20` reached.


(train_model pid=31429) 
(train_model pid=31429) 
(train_model pid=31429) 
(train_model pid=31429) 
(train_model pid=31494) 
(train_model pid=31429) 
(train_model pid=31494) 
Validation DataLoader 0:  16%|█▌        | 3/19 [00:00<00:00, 96.80it/s] 
(train_model pid=31494) 
(train_model pid=31494) 
(train_model pid=31494) 
(train_model pid=31494) 
(train_model pid=31494) 
(train_model pid=31494) 
Epoch 14:  99%|█████████▊| 73/74 [00:00<00:00, 81.32it/s, v_num=0, val_loss=0.169, val_r2=0.209] [repeated 4x across cluster]
(train_model pid=31494) 
(train_model pid=31494) 
Epoch 13:  65%|██████▍   | 48/74 [00:00<00:00, 48.57it/s, v_num=0, val_loss=2.010, val_r2=-4.51]
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31491) 
(train_model pid=31492) 
Epoch 16:  36%|███▋      | 27

(train_model pid=31492) `Trainer.fit` stopped: `max_epochs=20` reached. [repeated 4x across cluster]


Epoch 13: 100%|██████████| 74/74 [00:01<00:00, 53.44it/s, v_num=0, val_loss=1.630, val_r2=-1.09] [repeated 2x across cluster]
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
Epoch 15: 100%|██████████| 74/74 [00:01<00:00, 56.55it/s, v_num=0, val_loss=9.780, val_r2=-91.0]
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
(train_model pid=31491) 
Epoch 13:  42%|████▏     | 31/74 [00:00<00:00, 49.43it/s, v_num=0, val_loss=1.630, val_r2=-1.09] [repeated 4x across cluster]
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(train_model pid=31566) 
(

(train_model pid=31491) `Trainer.fit` stopped: `max_epochs=20` reached. [repeated 2x across cluster]
2026-06-15 13:55:51,699	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/home/franio/ray_results/train_model_2026-06-15_13-54-01' in 0.0091s.
2026-06-15 13:55:51,706	INFO tune.py:1033 -- Total run time: 110.75 seconds (110.67 seconds for the tuning loop).


In [102]:
print(analysis.get_best_config(metric='loss', mode='min'))

{'n_hidden': 8, 'hidden_size': 32, 'lr': 0.005076059720404389}
